In [2]:
!pip install igraph leidenalg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 82.1 MB/s eta 0:00:00


In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg as la
import community.community_louvain as community_louvain

from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
pd.set_option("display.max_columns", 100)

#### Применяем StandardScaler, чтобы привести все переменные в общий вид, также делаем некоторые проверки еще раз

In [4]:
df_unscaled = pd.read_csv("df_sberne.csv")
df = df_unscaled.copy()

cols = ['Средняя зарплата', 'population',
      'urban_share', 'share_industry', 'share_agriculture', 'share_services',
      'Здоровье_share', 'Маркетплейсы_share', 'Общественное питание_share',
    'Продовольствие_share', 'Транспорт_share']

scaler = StandardScaler().set_output(transform="pandas")
df[cols] = scaler.fit_transform(df[cols])

df['year']  = df['year'].astype(int)
df['month'] = df['month'].astype(int)

cats = ['Здоровье_share', 'Маркетплейсы_share',
        'Общественное питание_share', 'Продовольствие_share', 'Транспорт_share']

print("df:", df.shape)
print("Уникальные (year, month):")
print(df[['year', 'month']].drop_duplicates().sort_values(['year', 'month']).to_string(index=False))

df: (23915, 15)
Уникальные (year, month):
 year  month
 2023      1
 2023      2
 2023      3
 2023      4
 2023      5
 2023      6
 2023      7
 2023      8
 2023      9
 2023     10
 2023     11
 2023     12
 2024      1
 2024      2
 2024      3
 2024      4
 2024      5
 2024      6
 2024      7
 2024      8
 2024      9
 2024     10
 2024     11
 2024     12


## Начало эксперимента
Чтобы понять, какой способ проведения ребер лучше, проведем эксперимент - применим кластеризацию с каждым из методов для всех 24 методов и усредним метрики IVCI по ним, а затем решим какой из них лучше. Также переберем параметр resolution в louvain-кластеризации.

In [15]:
feature_cols = [
    'Здоровье_share',
    'Маркетплейсы_share',
    'Общественное питание_share',
    'Продовольствие_share',
    'Транспорт_share',
    'share_industry',
    'share_agriculture',
    'share_services',
    'urban_share',
    'Средняя зарплата',
    'population',
]

PERCENTILE_SIM = 75
K_MUTUAL       = 250
RESOLUTIONS    = [0.9, 1.0, 1.1]
TANIMOTO_EPS   = 1e-12

global_feature_min = (
    df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .min()
    .values
    .astype(float)
)

PERIODS = sorted(df[['year', 'month']].drop_duplicates().itertuples(index=False, name=None))
print(f"Всего периодов: {len(PERIODS)}")
print(PERIODS)

Всего периодов: 24
[(2023, 1), (2023, 2), (2023, 3), (2023, 4), (2023, 5), (2023, 6), (2023, 7), (2023, 8), (2023, 9), (2023, 10), (2023, 11), (2023, 12), (2024, 1), (2024, 2), (2024, 3), (2024, 4), (2024, 5), (2024, 6), (2024, 7), (2024, 8), (2024, 9), (2024, 10), (2024, 11), (2024, 12)]


Для сравнения выберем 4 наиболее перспективных метода:
- cosine similarity
- adaptive rbf
- tanimoto similarity
- mutual knn

In [16]:
def sim_cosine(X):
    S = cosine_similarity(X)
    np.fill_diagonal(S, 0)
    return S


def tanimoto_similarity(X):
    Xp = np.asarray(X, dtype=float)
    dot = Xp @ Xp.T
    sq = np.sum(Xp**2, axis=1)
    denom = sq[:, None] + sq[None, :] - dot
    S = np.divide(dot, denom, out=np.zeros_like(dot), where=denom > 0)
    np.fill_diagonal(S, 0)
    return S


def adaptive_rbf_similarity(X, k_local=7):
    n = X.shape[0]
    nn = NearestNeighbors(n_neighbors=min(k_local + 1, n)).fit(X)
    dist, _ = nn.kneighbors(X)
    sigma = np.maximum(dist[:, -1], 1e-9)
    D2 = euclidean_distances(X, squared=True)
    denom = np.outer(sigma, sigma)
    W = np.exp(-D2 / denom)
    np.fill_diagonal(W, 0)
    W[W<1e-3]=0
    return W

def nx_to_igraph(G):
    mapping = {n: i for i, n in enumerate(G.nodes())}
    inv_mapping = {i: n for n, i in mapping.items()}
    edges = [(mapping[u], mapping[v]) for u, v in G.edges()]
    weights = [G[u][v].get('weight', 1.0) for u, v in G.edges()]
    g = ig.Graph(n=G.number_of_nodes(), edges=edges, directed=False)
    g.es['weight'] = weights
    g.vs['name'] = list(G.nodes())
    return g, inv_mapping


In [17]:
def build_threshold_graph_percentile_sim(sim, percentile=75):
    n = sim.shape[0]
    S = sim.copy()
    np.fill_diagonal(S, 0)

    iu = np.triu_indices_from(S, k=1)
    vals = S[iu]
    pos = vals[vals > 0]
    if len(pos) == 0:
        return nx.Graph(), np.nan
    thr = np.percentile(pos, percentile)

    A = np.zeros((n, n), dtype=bool)
    A[iu] = vals >= thr
    A = A | A.T

    G = nx.Graph()
    G.add_nodes_from(range(n))
    ii, jj = np.where(np.triu(A, k=1))
    for i, j in zip(ii, jj):
        G.add_edge(int(i), int(j), weight=float(S[i, j]))
    return G, float(thr)


def build_mutual_knn_graph(X, k=250):
    n = X.shape[0]
    k_eff = min(k, n - 1)
    nn = NearestNeighbors(n_neighbors=k_eff + 1).fit(X)
    dist, idx = nn.kneighbors(X)
    idx = idx[:, 1:]
    dist = dist[:, 1:]

    neighbor_sets = [set(idx[i].tolist()) for i in range(n)]
    sigma = np.maximum(dist[:, -1], 1e-9)

    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in neighbor_sets[i]:
            if i < j and i in neighbor_sets[j]:
                d2 = float(np.sum((X[i] - X[j]) ** 2))
                w = float(np.exp(-d2 / (sigma[i] * sigma[j] + 1e-12)))
                G.add_edge(i, j, weight=w)

    for i in range(n):
        if G.degree(i) == 0:
            j = int(idx[i, 0])
            d2 = float(np.sum((X[i] - X[j]) ** 2))
            w = float(np.exp(-d2 / (sigma[i] * sigma[j] + 1e-12)))
            G.add_edge(i, j, weight=w)
    return G


def graph_stats(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    return {"n_nodes": n, "n_edges": m,
            "avg_degree": round(2 * m / n, 2) if n > 0 else 0}

In [18]:
def leiden_partition(G, seed=42, resolution=1.0):
    if G.number_of_nodes() == 0:
        return np.array([])
    g, inv_mapping = nx_to_igraph(G)
    part = la.find_partition(
        g,
        la.RBConfigurationVertexPartition,
        weights='weight',
        resolution_parameter=resolution,
        seed=seed,
    )
    n = G.number_of_nodes()
    return np.array(part.membership)


def louvain_partition(G, seed=42, resolution=1.0):
    if G.number_of_nodes() == 0:
        return np.array([])

    nodes = list(G.nodes())
    mapping = {n: i for i, n in enumerate(nodes)}
    inv_mapping = {i: n for n, i in mapping.items()}

    G_relabeled = nx.relabel_nodes(G, mapping)
    G_relabeled = nx.Graph(G_relabeled)  # на всякий случай
    G_relabeled.add_nodes_from(range(len(nodes)))

    partition_dict = community_louvain.best_partition(
        G_relabeled,
        weight='weight',
        resolution=resolution,
        random_state=seed,
    )

    labels = np.array([partition_dict[inv_mapping[i]] for i in range(len(nodes))])
    return labels


ALGO_FUNCS = {
    "leiden":  leiden_partition,
    "louvain": louvain_partition,
}


def cluster_sizes(labels):
    return pd.Series(labels).value_counts().sort_index()

#### Задаем IVCI-метрики

In [19]:
def s_dbw(X, labels):
    labs = np.unique(labels)
    k = len(labs)
    if k < 2:
        return np.nan

    centroids = np.array([X[labels == lab].mean(axis=0) for lab in labs])
    cluster_std = np.array([
        np.std(X[labels == lab], axis=0, ddof=0)
        for lab in labs
    ])
    global_std = np.std(X, axis=0, ddof=0)

    global_std_norm = np.linalg.norm(global_std)
    if global_std_norm <= 0:
        return np.nan

    scat = np.mean(np.linalg.norm(cluster_std, axis=1)) / global_std_norm
    stdev = global_std_norm

    def density(center, indices):
        if len(indices) == 0:
            return 0
        d = np.linalg.norm(X[indices] - center, axis=1)
        return int(np.sum(d <= stdev))

    dens_centers = np.array([
        density(centroids[i], np.where(labels == labs[i])[0])
        for i in range(k)
    ], dtype=float)

    dens_bw = 0.0
    pair_count = 0

    for i in range(k):
        idx_i = np.where(labels == labs[i])[0]
        for j in range(i + 1, k):
            idx_j = np.where(labels == labs[j])[0]

            midpoint = 0.5 * (centroids[i] + centroids[j])
            dens_mid = density(midpoint, np.concatenate([idx_i, idx_j]))

            denom = max(dens_centers[i], dens_centers[j])
            if denom > 0:
                dens_bw += dens_mid / denom

            pair_count += 1

    if pair_count > 0:
        dens_bw /= pair_count

    return float(scat + dens_bw)


def avi_score(G, partition_dict):
    nodes = list(G.nodes())
    if len(nodes) == 0:
        return np.nan

    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight", dtype=float)
    labels = np.array([partition_dict[n] for n in nodes])
    labs = np.unique(labels)

    values = []
    for lab in labs:
        mask = labels == lab
        internal = A[np.ix_(mask, mask)].sum()
        external = A[np.ix_(mask, ~mask)].sum()
        denom = internal + external
        values.append(internal / denom if denom > 0 else np.nan)

    return float(np.nanmean(values))


def avu_score(G, partition_dict):
    nodes = list(G.nodes())
    if len(nodes) == 0:
        return np.nan

    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight", dtype=float)
    labels = np.array([partition_dict[n] for n in nodes])
    labs = np.unique(labels)
    k = len(labs)

    if k < 2:
        return np.nan

    ext = {}
    for lab in labs:
        mask = labels == lab
        ext[lab] = A[np.ix_(mask, ~mask)].sum()

    total = 0.0
    for lab_k in labs:
        mask_k = labels == lab_k
        for lab_l in labs:
            if lab_l == lab_k:
                continue

            mask_l = labels == lab_l
            between = A[np.ix_(mask_k, mask_l)].sum()
            denom = ext[lab_k] + ext[lab_l] - between

            if denom > 0:
                total += between / denom

    return float(total / k)


def mq_score(G, partition_dict):
    nodes = list(G.nodes())
    if len(nodes) == 0:
        return np.nan

    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight", dtype=float)
    labels = np.array([partition_dict[n] for n in nodes])
    labs = np.unique(labels)
    k = len(labs)

    if k == 1:
        mask = labels == labs[0]
        n = int(mask.sum())
        if n < 2:
            return np.nan
        internal = A[np.ix_(mask, mask)].sum() / 2.0
        return float(internal / (n * (n - 1) / 2.0))

    intra = []
    for lab in labs:
        mask = labels == lab
        n_i = int(mask.sum())
        if n_i < 2:
            intra.append(0.0)
            continue

        internal = A[np.ix_(mask, mask)].sum() / 2.0
        max_internal = n_i * (n_i - 1) / 2.0
        intra.append(internal / max_internal)

    inter = []
    for i, lab_i in enumerate(labs):
        mask_i = labels == lab_i
        n_i = int(mask_i.sum())

        for lab_j in labs[i + 1:]:
            mask_j = labels == lab_j
            n_j = int(mask_j.sum())

            between = A[np.ix_(mask_i, mask_j)].sum()
            max_between = n_i * n_j

            inter.append(
                between / max_between if max_between > 0 else 0.0
            )

    mean_inter = float(np.mean(inter)) if inter else 0.0
    return float(np.mean(intra) - mean_inter)


def evaluate_clustering_graph(G, X_eval, partition_dict, name="", n_obs=None):
    nodes = list(G.nodes())
    labels = np.array([partition_dict[n] for n in nodes])
    n_lab = len(np.unique(labels))

    if n_obs is None:
        n_obs = len(X_eval)

    res = {"method": name, "n_clusters": n_lab}

    if n_lab < 2:
        return {
            **res,
            "SW": np.nan,
            "CH": np.nan,
            "S_Dbw": np.nan,
            "AVI": np.nan,
            "AVU": np.nan,
            "MQ": np.nan,
        }

    res["SW"] = silhouette_score(X_eval, labels)

    res["CH"] = calinski_harabasz_score(X_eval, labels) / n_obs

    res["S_Dbw"] = s_dbw(X_eval, labels)
    res["AVI"] = avi_score(G, partition_dict)
    res["AVU"] = avu_score(G, partition_dict)
    res["MQ"] = mq_score(G, partition_dict)

    return res


In [20]:
def process_period(year, month):
    sub = df[(df['year'] == year) & (df['month'] == month)].copy()
    sub = sub.dropna(subset=feature_cols)
    sub['urban_share'] = sub['urban_share'].replace([np.inf, -np.inf], 0)
    sub = sub.reset_index(drop=True)

    n_obs = len(sub)
    if n_obs < 10:
        return []

    X = sub[feature_cols].values.astype(float)
    X_tan = X - global_feature_min + TANIMOTO_EPS

    S_cos   = sim_cosine(X)
    S_tan   = tanimoto_similarity(X_tan)
    S_adrbf = adaptive_rbf_similarity(X, k_local=7)

    G_cos, _ = build_threshold_graph_percentile_sim(S_cos, percentile=PERCENTILE_SIM)
    G_tan, _ = build_threshold_graph_percentile_sim(S_tan, percentile=PERCENTILE_SIM)
    G_ar, _  = build_threshold_graph_percentile_sim(S_adrbf, percentile=PERCENTILE_SIM)
    G_mut    = build_mutual_knn_graph(X, k=K_MUTUAL)

    GRAPHS_LOCAL = {
        "cosine":       G_cos,
        "tanimoto":     G_tan,
        "mutual_knn":   G_mut,
        "adaptive_rbf": G_ar,
    }

    rows_local = []

    for method, G in GRAPHS_LOCAL.items():
        for algo_name, algo_func in ALGO_FUNCS.items():
            for res in RESOLUTIONS:
                labels_arr = algo_func(G, seed=42, resolution=res)
                part = {n: int(labels_arr[i]) for i, n in enumerate(G.nodes())}

                row = evaluate_clustering_graph(G, X, part, method, n_obs=n_obs)
                row["algorithm"]  = algo_name
                row["resolution"] = res
                row["year"]       = year
                row["month"]      = month
                row["n_obs"]      = n_obs
                rows_local.append(row)

    return rows_local

#### Сам эксперимент

In [ ]:
all_rows = []
for (y, m) in PERIODS:
    rows = process_period(y, m)
    all_rows.extend(rows)
    print(f"  {y}-{m:02d}: {len(rows)} строк, "
          f"n_obs={rows[0]['n_obs'] if rows else 0}")

raw_df = pd.DataFrame(all_rows)
print("\nraw_df:", raw_df.shape)
raw_df.head()

  2023-01: 24 строк, n_obs=1027
  2023-02: 24 строк, n_obs=1027
  2023-03: 24 строк, n_obs=1027
  2023-04: 24 строк, n_obs=1021
  2023-05: 24 строк, n_obs=1019
  2023-06: 24 строк, n_obs=1020
  2023-07: 24 строк, n_obs=1020
  2023-08: 24 строк, n_obs=1018
  2023-09: 24 строк, n_obs=1015
  2023-10: 24 строк, n_obs=1015
  2023-11: 24 строк, n_obs=1012
  2023-12: 24 строк, n_obs=1014
  2024-01: 24 строк, n_obs=1010
  2024-02: 24 строк, n_obs=1012
  2024-03: 24 строк, n_obs=1011
  2024-04: 24 строк, n_obs=961
  2024-05: 24 строк, n_obs=961
  2024-06: 24 строк, n_obs=958
  2024-07: 24 строк, n_obs=961
  2024-08: 24 строк, n_obs=963
  2024-09: 24 строк, n_obs=966
  2024-10: 24 строк, n_obs=956
  2024-11: 24 строк, n_obs=958
  2024-12: 24 строк, n_obs=963

raw_df: (576, 13)


,method,n_clusters,SW,CH,S_Dbw,AVI,AVU,MQ,algorithm,resolution,year,month,n_obs
0,cosine,4,0.149854,0.147946,1.994245,0.620036,0.555413,0.334357,leiden,0.9,2023,1,1027
1,cosine,4,0.144173,0.149783,2.236007,0.647134,0.594926,0.271854,leiden,1.0,2023,1,1027
2,cosine,6,0.043074,0.112685,1.810720,0.494247,0.423621,0.219732,leiden,1.1,2023,1,1027
3,cosine,6,0.121898,0.099807,1.890164,0.496065,0.400107,0.338313,louvain,0.9,2023,1,1027
4,cosine,5,0.152417,0.142818,2.066659,0.597541,0.496066,0.311659,louvain,1.0,2023,1,1027


In [ ]:
metric_cols = ["SW", "CH", "S_Dbw", "AVI", "AVU", "MQ", "n_clusters"]

agg_df = (
    raw_df
    .groupby(["algorithm", "method", "resolution"])[metric_cols]
    .mean()
    .reset_index()
    .sort_values(["algorithm", "method", "resolution"])
)

In [ ]:
from IPython.display import display, HTML

DIRECTIONS = {
    "SW":    True,
    "CH":    True,
    "S_Dbw": False,
    "AVI":   True,
    "AVU":   False,
    "MQ":    True,
}

C1, BG1 = "#7ee787", "#0f3d1a"
C2, BG2 = "#ffd33d", "#4a3a00"
C3, BG3 = "#ff7b72", "#4a1512"

def color_top3(series, higher_is_better=True, decimals=4):
    s = series.copy()
    order = s.sort_values(ascending=not higher_is_better).index
    top3 = list(order[:3])

    colors = [
        (C1, BG1),
        (C2, BG2),
        (C3, BG3),
    ]

    out = pd.Series("", index=s.index)
    for idx in s.index:
        val = f"{s.loc[idx]:.{decimals}f}"
        if idx in top3:
            pos = top3.index(idx)
            c, bg = colors[pos]
            out.loc[idx] = (
                f'<span style="color:{c}; background:{bg}; '
                f'font-weight:bold; padding:2px 6px; border-radius:4px;">'
                f'{val}</span>'
            )
        else:
            out.loc[idx] = val
    return out


tab = agg_df.copy()

formatted = pd.DataFrame()
formatted["algorithm"]  = tab["algorithm"]
formatted["method"]     = tab["method"]
formatted["resolution"] = tab["resolution"]
formatted["k"]          = tab["n_clusters"].round(2)

for metric, higher_better in DIRECTIONS.items():
    formatted[metric] = color_top3(tab[metric], higher_is_better=higher_better)

html = formatted.to_html(index=False, escape=False, justify="center")

html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse:collapse; text-align:center; '
    'font-family:Arial; font-size:13px; '
    'background:#1e1e1e; color:#e6e6e6;" border="1">'
)
html = html.replace(
    '<th>',
    '<th style="padding:7px 12px; background:#333333; color:#ffffff; '
    'border:1px solid #555; font-weight:bold;">'
)
html = html.replace(
    '<td>',
    '<td style="padding:5px 10px; border:1px solid #444; color:#e6e6e6;">'
)

legend = (
    '<div style="font-family:Arial; font-size:13px; margin:8px 0; color:#e6e6e6;">'
    f'<span style="background:{BG1}; color:{C1}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-1</span> &nbsp; '
    f'<span style="background:{BG2}; color:{C2}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-2</span> &nbsp; '
    f'<span style="background:{BG3}; color:{C3}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-3</span>'
    '</div>'
)

display(HTML(legend + html))

algorithm,method,resolution,k,SW,CH,S_Dbw,AVI,AVU,MQ
leiden,adaptive_rbf,0.9,5.42,0.1355,0.1676,1.9729,0.6857,0.5198,0.0841
leiden,adaptive_rbf,1.0,5.88,0.1211,0.1578,2.0371,0.6569,0.5252,0.0845
leiden,adaptive_rbf,1.1,7.00,0.1150,0.1476,2.0501,0.6155,0.5138,0.0796
leiden,cosine,0.9,3.62,0.1493,0.1872,2.2128,0.7618,0.6371,0.2585
leiden,cosine,1.0,4.12,0.1491,0.1792,2.1980,0.7277,0.6405,0.2689
leiden,cosine,1.1,4.71,0.1429,0.1703,2.1472,0.6860,0.6186,0.2841
leiden,mutual_knn,0.9,3.67,0.1420,0.1852,2.2233,0.7240,0.6164,0.1931
leiden,mutual_knn,1.0,4.25,0.1371,0.1711,2.1544,0.6796,0.5905,0.2116
leiden,mutual_knn,1.1,4.62,0.1281,0.1656,2.1483,0.6550,0.5809,0.2182
leiden,tanimoto,0.9,7.88,0.1498,0.0920,0.8762,0.6813,0.2702,0.3471


Видим, что Tanimoto выигрывает по многим метрикам, однако он строит много кластеров, а это экономически не интерпретируемо. Так что ради интерпретируемости мы пожертвуем качеством и не будем использовать Tanimoto.

In [ ]:
from IPython.display import display, HTML

DIRECTIONS = {
    "SW":    True,
    "CH":    True,
    "S_Dbw": False,
    "AVI":   True,
    "AVU":   False,
    "MQ":    True,
}

C1, BG1 = "#7ee787", "#0f3d1a"
C2, BG2 = "#ffd33d", "#4a3a00"
C3, BG3 = "#ff7b72", "#4a1512"

def color_top3(series, higher_is_better=True, decimals=4):
    s = series.copy()
    order = s.sort_values(ascending=not higher_is_better).index
    top3 = list(order[:3])

    colors = [
        (C1, BG1),
        (C2, BG2),
        (C3, BG3),
    ]

    out = pd.Series("", index=s.index)

    for idx in s.index:
        val = f'{s.loc[idx]:.{decimals}f}'

        if idx in top3:
            pos = top3.index(idx)
            c, bg = colors[pos]

            out.loc[idx] = (
                f'<span style="color:{c}; background:{bg}; '
                f'font-weight:bold; padding:2px 6px; border-radius:4px;">'
                f'{val}</span>'
            )
        else:
            out.loc[idx] = val

    return out


tab = agg_df[agg_df["method"] != "tanimoto"].copy()

formatted = pd.DataFrame()
formatted["algorithm"]  = tab["algorithm"]
formatted["method"]     = tab["method"]
formatted["resolution"] = tab["resolution"]
formatted["k"]          = tab["n_clusters"].round(2)

for metric, higher_better in DIRECTIONS.items():
    formatted[metric] = color_top3(tab[metric], higher_is_better=higher_better)

for metric, higher_better in DIRECTIONS.items():
    formatted[metric] = color_top3(
        tab[metric],
        higher_is_better=higher_better
    )

html = formatted.to_html(
    index=False,
    escape=False,
    justify="center"
)

html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse:collapse; text-align:center; '
    'font-family:Arial; font-size:13px; '
    'background:#1e1e1e; color:#e6e6e6;" border="1">'
)

html = html.replace(
    '<th>',
    '<th style="padding:7px 12px; background:#333333; color:#ffffff; '
    'border:1px solid #555; font-weight:bold;">'
)

html = html.replace(
    '<td>',
    '<td style="padding:5px 10px; border:1px solid #444; color:#e6e6e6;">'
)

legend = (
    '<div style="font-family:Arial; font-size:13px; margin:8px 0; color:#e6e6e6;">'
    f'<span style="background:{BG1}; color:{C1}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-1</span> &nbsp; '
    f'<span style="background:{BG2}; color:{C2}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-2</span> &nbsp; '
    f'<span style="background:{BG3}; color:{C3}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-3</span>'
    '</div>'
)

display(HTML(legend + html))

algorithm,method,resolution,k,SW,CH,S_Dbw,AVI,AVU,MQ
leiden,adaptive_rbf,0.9,5.42,0.1355,0.1676,1.9729,0.6857,0.5198,0.0841
leiden,adaptive_rbf,1.0,5.88,0.1211,0.1578,2.0371,0.6569,0.5252,0.0845
leiden,adaptive_rbf,1.1,7.00,0.1150,0.1476,2.0501,0.6155,0.5138,0.0796
leiden,cosine,0.9,3.62,0.1493,0.1872,2.2128,0.7618,0.6371,0.2585
leiden,cosine,1.0,4.12,0.1491,0.1792,2.1980,0.7277,0.6405,0.2689
leiden,cosine,1.1,4.71,0.1429,0.1703,2.1472,0.6860,0.6186,0.2841
leiden,mutual_knn,0.9,3.67,0.1420,0.1852,2.2233,0.7240,0.6164,0.1931
leiden,mutual_knn,1.0,4.25,0.1371,0.1711,2.1544,0.6796,0.5905,0.2116
leiden,mutual_knn,1.1,4.62,0.1281,0.1656,2.1483,0.6550,0.5809,0.2182
louvain,adaptive_rbf,0.9,6.46,0.1317,0.1552,1.8798,0.6605,0.4947,0.1093


In [ ]:
tab = agg_df[agg_df["method"] != "tanimoto"].copy()

### Смотрим на корелляции между метриками

In [ ]:
tab = tab[tab.method!="tanimoto"]

In [ ]:
columns = ['method', 'resolution', 'SW', 'CH', 'S_Dbw', 'AVI', 'AVU', 'MQ']

metrics = ['SW', 'CH', 'S_Dbw', 'AVI', 'AVU', 'MQ']

corr_spearman = tab[metrics].corr(method='spearman')

corr_spearman.round(3)

,SW,CH,S_Dbw,AVI,AVU,MQ
SW,1.000,0.900,0.767,0.133,0.933,0.817
CH,0.900,1.000,0.900,0.183,0.817,0.583
S_Dbw,0.767,0.900,1.000,-0.050,0.800,0.550
AVI,0.133,0.183,-0.050,1.000,0.000,-0.283
AVU,0.933,0.817,0.800,0.000,1.000,0.883
MQ,0.817,0.583,0.550,-0.283,0.883,1.000


### Корелляция очень большая, поэтому странно было бы учитывать их все на равных
#### Используем метод CRITIC, который учитывает различность метрик.

In [ ]:
metrics = [
    'SW',
    'CH',
    'S_Dbw',
    'AVI',
    'AVU',
    'MQ'
]

higher_is_better = {
    'SW': True,
    'CH': True,
    'S_Dbw': False,
    'AVI': True,
    'AVU': False,
    'MQ': True
}

In [ ]:
norm_df = pd.DataFrame(index=tab.index)

for metric in metrics:
    x = tab[metric]

    if higher_is_better[metric]:
        norm_df[metric] = (x - x.min()) / (x.max() - x.min())
    else:
        norm_df[metric] = (x.max() - x) / (x.max() - x.min())

In [ ]:
std = norm_df.std()

corr = norm_df.corr(method='spearman')

In [ ]:
critic_score = {}

for metric in metrics:
    conflict = sum(
        1 - corr.loc[metric, other]
        for other in metrics
        if other != metric
    )

    critic_score[metric] = std[metric] * conflict

critic_score = pd.Series(critic_score)

weights = critic_score / critic_score.sum()

print("Веса метрик:")
print(weights.sort_values(ascending=False).round(3))

Веса метрик:
AVU      0.258
S_Dbw    0.199
MQ       0.168
SW       0.131
CH       0.130
AVI      0.114
dtype: float64


In [ ]:
borda = pd.DataFrame(index=tab.index)

n = len(tab)

for metric in metrics:

    if higher_is_better[metric]:
        rank = tab[metric].rank(
            ascending=False,
            method='average'
        )
    else:
        rank = tab[metric].rank(
            ascending=True,
            method='average'
        )

    borda[metric] = n - rank

In [ ]:
weighted_borda = pd.Series(0.0, index=tab.index)

for metric in metrics:
    weighted_borda += borda[metric] * weights[metric]

In [ ]:
ranking = tab[['algorithm', 'method', 'resolution']].copy()
ranking['Borda_score'] = weighted_borda
ranking = ranking.sort_values('Borda_score', ascending=False).reset_index(drop=True)
ranking

,algorithm,method,resolution,Borda_score
0,louvain,cosine,0.9,10.963570
1,louvain,adaptive_rbf,0.9,10.584393
2,louvain,mutual_knn,0.9,10.282420
3,leiden,adaptive_rbf,0.9,10.107124
4,louvain,cosine,1.1,9.521770
5,leiden,cosine,1.1,9.254595
6,louvain,cosine,1.0,8.981410
7,leiden,cosine,0.9,8.716075
8,leiden,cosine,1.0,8.320641
9,louvain,adaptive_rbf,1.1,8.098480


#### Проверим стабильность метода на разных сидах

In [21]:
SEEDS = [0, 1, 7, 42, 123, 2024]
RES = 0.9

all_seed_rows = []

for seed in SEEDS:
    for (y, m) in PERIODS:
        sub = df[(df['year'] == y) & (df['month'] == m)].copy()
        sub = sub.dropna(subset=feature_cols)
        sub['urban_share'] = sub['urban_share'].replace([np.inf, -np.inf], 0)
        sub = sub.reset_index(drop=True)
        if len(sub) < 10:
            continue

        X = sub[feature_cols].values.astype(float)
        G, _ = build_threshold_graph_percentile_sim(sim_cosine(X), PERCENTILE_SIM)

        lv = louvain_partition(G, seed=seed, resolution=RES)
        part = {n: int(lv[i]) for i, n in enumerate(G.nodes())}

        row = evaluate_clustering_graph(G, X, part, "louvain", n_obs=len(sub))
        row["seed"]   = seed
        row["year"]   = y
        row["month"]  = m
        row["n_obs"]  = len(sub)
        all_seed_rows.append(row)

seed_df = pd.DataFrame(all_seed_rows)
print("seed_df:", seed_df.shape)
print(seed_df.groupby("seed")["n_clusters"].agg(["mean", "std", "min", "max"]).round(2))

seed_df: (144, 12)
      mean   std  min  max
seed                      
0     4.92  0.83    3    6
1     4.62  0.97    3    7
7     4.71  0.69    4    6
42    4.92  0.58    4    6
123   4.71  0.86    3    7
2024  4.83  0.87    3    7


In [22]:
metric_cols = ["SW", "CH", "S_Dbw", "AVI", "AVU", "MQ", "n_clusters"]

seed_summary = (
    seed_df
    .groupby("seed")[metric_cols]
    .mean()
    .round(4)
)
seed_summary

,SW,CH,S_Dbw,AVI,AVU,MQ,n_clusters
seed,,,,,,,
0,0.1465,0.1590,2.0451,0.6690,0.5783,0.3045,4.9167
1,0.1438,0.1631,2.0548,0.6833,0.5845,0.2815,4.6250
7,0.1483,0.1621,2.0713,0.6815,0.5875,0.2909,4.7083
42,0.1544,0.1641,2.0526,0.6727,0.5854,0.3000,4.9167
123,0.1492,0.1670,2.0821,0.6867,0.5971,0.3003,4.7083
2024,0.1515,0.1691,2.0809,0.6837,0.5979,0.2995,4.8333


In [30]:
within_mean = seed_df.groupby("seed")[metric_cols].mean()
within_std  = seed_df.groupby("seed")[metric_cols].std()

cv = (within_std / within_mean.abs()).round(3)
cv

,SW,CH,S_Dbw,AVI,AVU,MQ,n_clusters
seed,,,,,,,
0,0.122,0.131,0.068,0.090,0.104,0.122,0.169
1,0.207,0.142,0.073,0.101,0.122,0.143,0.210
7,0.132,0.141,0.057,0.076,0.108,0.111,0.147
42,0.090,0.122,0.044,0.081,0.096,0.093,0.119
123,0.130,0.151,0.067,0.083,0.097,0.131,0.182
2024,0.128,0.166,0.059,0.099,0.128,0.117,0.180
